In [1]:
import os
import pandas as pd
from glob import iglob

In [2]:
metadata = pd.read_csv('/shares/CIBIO-Storage/CM/scratch/users/aitor.blancomiguez/analyses/nw_metanalysis/nw_metadata.tsv', sep='\t')
metadata.index = metadata['sample_id']
metadata

,study_name,sample_id,subject_id,body_site,antibiotics_current_use,study_condition,disease,age,age_category,gender,...,non_westernized,sequencing_platform,PMID,number_reads,number_bases,minimum_read_length,median_read_length,NCBI_accession,curator,DNA_extraction_kit
sample_id,,,,,,,,,,,,,,,,,,,,,
O2_UC49_0,NielsenHB_2014,O2_UC49_0,O2_UC49,stool,NaN,control,healthy,22.0,adult,female,...,no,IlluminaHiSeq,24997787.0,60584637.0,4.099972e+09,30.0,70.0,ERR210493;ERR210492;ERR209641;ERR209640,Paolo_Manghi,NaN
O2_UC49_2,NielsenHB_2014,O2_UC49_2,O2_UC49,stool,NaN,control,healthy,22.0,adult,female,...,no,IlluminaHiSeq,24997787.0,56508214.0,3.775477e+09,30.0,70.0,ERR210495;ERR210494;ERR209643;ERR209642,Paolo_Manghi,NaN
O2_UC50_0,NielsenHB_2014,O2_UC50_0,O2_UC50,stool,NaN,control,healthy,24.0,adult,male,...,no,IlluminaHiSeq,24997787.0,65105855.0,4.392995e+09,30.0,71.0,ERR210497;ERR210496;ERR209645;ERR209644,Paolo_Manghi,NaN
O2_UC50_2,NielsenHB_2014,O2_UC50_2,O2_UC50,stool,NaN,control,healthy,24.0,adult,male,...,no,IlluminaHiSeq,24997787.0,53260679.0,3.596382e+09,30.0,70.0,ERR210500;ERR209648,Paolo_Manghi,NaN
O2_UC51_0,NielsenHB_2014,O2_UC51_0,O2_UC51,stool,NaN,control,healthy,32.0,adult,female,...,no,IlluminaHiSeq,24997787.0,57486896.0,3.923766e+09,30.0,72.0,ERR210502;ERR210501;ERR209650;ERR209649,Paolo_Manghi,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SAMEA6415059,BorryM_2020,SAMEA6415059,SAMEA6415059,stool,NaN,control,healthy,NaN,NaN,NaN,...,ancient,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ERR3761407,BorryM_2020,ERR3761407,ERR3761407,stool,NaN,control,healthy,NaN,NaN,NaN,...,ancient,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ERR3761411,BorryM_2020,ERR3761411,ERR3761411,stool,NaN,control,healthy,NaN,NaN,NaN,...,ancient,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
to_profile_sgbs = dict()
prevalence_threshold = 0.05
with open('/shares/CIBIO-Storage/CM/scratch/users/aitor.blancomiguez/analyses/nw_metanalysis/strainphlan/print_clades_only.txt', 'r') as read_file:
    for line in read_file:
        if 't__SGB' in line:
            sgb = line.strip().split('t__')[1].split(':')[0]
            samples = line.strip().split('[')[1].split(']')[0].replace("'",'').split(', ')
            nw = 0
            w = 0
            for sample in samples:
                if metadata.loc[sample]['non_westernized'] == 'yes':
                    nw += 1
                elif metadata.loc[sample]['non_westernized'] == 'no':
                    w += 1
            if nw / len(metadata[metadata['non_westernized'] == 'yes']) >= prevalence_threshold or w / len(metadata[metadata['non_westernized'] == 'no']) >= prevalence_threshold:
                to_profile_sgbs[sgb] = samples

In [4]:
len(to_profile_sgbs)

347

In [5]:
to_profile_sgbs

124

In [6]:
to_profile_sgbs.keys()

dict_keys(['SGB4933_group', 'SGB1836_group', 'SGB1814', 'SGB4837_group', 'SGB4874', 'SGB14546_group', 'SGB15316_group', 'SGB2318', 'SGB15286', 'SGB15342', 'SGB15332_group', 'SGB15318_group', 'SGB4925', 'SGB4285_group', 'SGB17248', 'SGB4540_group', 'SGB1934', 'SGB15254', 'SGB4563_group', 'SGB17244', 'SGB5082_group', 'SGB15300', 'SGB4581', 'SGB15346', 'SGB4532', 'SGB1871', 'SGB4577_group', 'SGB1949', 'SGB4914', 'SGB4262', 'SGB1877', 'SGB1626', 'SGB4826_group', 'SGB4940', 'SGB4820', 'SGB10068', 'SGB2295', 'SGB15368', 'SGB1861', 'SGB1790', 'SGB5090_group', 'SGB4811_group', 'SGB15106', 'SGB9226', 'SGB5075_group', 'SGB15053_group', 'SGB14861', 'SGB4951', 'SGB4910', 'SGB15249', 'SGB4868', 'SGB4993', 'SGB4575', 'SGB8007_group', 'SGB5117', 'SGB15317', 'SGB6796_group', 'SGB5111', 'SGB4348', 'SGB6754', 'SGB4303', 'SGB4557', 'SGB1855_group', 'SGB15265_group', 'SGB4198_group', 'SGB4705', 'SGB4828_group', 'SGB17256', 'SGB714_group', 'SGB4582_group', 'SGB14993_group', 'SGB4367', 'SGB15234', 'SGB4936'

In [12]:
for sgb in to_profile_sgbs:
    os.mkdir('/shares/CIBIO-Storage/CM/scratch/users/aitor.blancomiguez/analyses/nw_metanalysis/strainphlan/sgb_sample2markers/{}'.format(sgb))
    for sample in to_profile_sgbs[sgb]:
        dataset = metadata.loc[sample]['study_name']
        org = '/shares/CIBIO-Storage/CM/scratch/users/aitor.blancomiguez/analyses/nw_metanalysis/strainphlan/sample2markers/{}/{}.pkl'.format(dataset, sample)
        dest = '/shares/CIBIO-Storage/CM/scratch/users/aitor.blancomiguez/analyses/nw_metanalysis/strainphlan/sgb_sample2markers/{}/{}.pkl'.format(sgb,sample)
        os.symlink(org, dest)

In [7]:
for sgb in to_profile_sgbs:
    if not os.path.exists('/shares/CIBIO-Storage/CM/scratch/users/aitor.blancomiguez/analyses/nw_metanalysis/strainphlan/sgb_sample2markers/{}'.format(sgb)):
        os.mkdir('/shares/CIBIO-Storage/CM/scratch/users/aitor.blancomiguez/analyses/nw_metanalysis/strainphlan/sgb_sample2markers_2/{}'.format(sgb))
        for sample in to_profile_sgbs[sgb]:
            dataset = metadata.loc[sample]['study_name']
            org = '/shares/CIBIO-Storage/CM/scratch/users/aitor.blancomiguez/analyses/nw_metanalysis/strainphlan/sample2markers/{}/{}.pkl'.format(dataset, sample)
            dest = '/shares/CIBIO-Storage/CM/scratch/users/aitor.blancomiguez/analyses/nw_metanalysis/strainphlan/sgb_sample2markers_2/{}/{}.pkl'.format(sgb,sample)
            os.symlink(org, dest)

### Get graphlan metadata

In [6]:
for tree in iglob('/shares/CIBIO-Storage/CM/scratch/users/aitor.blancomiguez/analyses/nw_metanalysis/strainphlan/output/SGB4925/RAxML_bestTree.t__SGB*.StrainPhlAn3.tre'):
    tree_n = list()
    with open(tree.replace(tree.split('/')[-1], 'graphlan_metadata2.tsv'), 'w') as write_file:
        with open(tree, 'r') as read_file:
            m = read_file.readline()
            m = m.split(',')
            for n in m:
                if "'" in n:
                    tree_n.append(n.split(":")[0].split("(")[-1].split("'")[1])
                else:
                    tree_n.append(n.split(":")[0].split("(")[-1])
            for line in tree_n:
                line2 = line.replace(".", ",")
                write_file.write(line2+"\tclade_marker_edge_width\t0.2\n")

                if line in metadata['sample_id'].values.tolist():
                    meta = metadata[metadata['sample_id'] == line]
                    
                    if meta['non_westernized'].values[0] == 'NHP':
                        if line.split('__')[0] in ['LiX_2018','SrivathsanA_2015']:
                            write_file.write(line2+"\tclade_marker_color\t#FC4E07\n")
                        else:
                            write_file.write(line2+"\tclade_marker_color\t#00AFBB\n")
                            
                        write_file.write(line2+"\tclade_marker_shape\t*\n") 
                        write_file.write(line2+"\tclade_marker_size\t50\n")
            
                    
                    elif meta['non_westernized'].values[0] == 'ancient':
                        write_file.write(line2+"\tclade_marker_color\t#E7B800\n")                        
                        write_file.write(line2+"\tclade_marker_shape\t*\n") 
                        write_file.write(line2+"\tclade_marker_size\t50\n")                    
                    else:
                        write_file.write(line2+"\tclade_marker_size\t0\n")
                        if meta['non_westernized'].values[0] == 'yes':
                            write_file.write(line2+"\tring_color\t2\t#dede0e\n")
                        else:
                            write_file.write(line2+"\tring_color\t2\t#2b8cbe\n")
                        country = meta['country'].values[0]
                        
                        if country in ['ARG', 'COL', 'PER', 'SLV']:
                            color = '#72b6a1'
                        elif country in ['CHN', 'JPN', 'IND', 'KOR', 'MNG', 'IDN', 'ISR', 'BGD']:
                            color = '#e99675'
                        elif country in['ETH', 'GUI', 'MDG', 'TZA', 'GHA', 'CMR', 'LBR', 'GNB']:
                            color = '#e5c849'
                        elif country in ['FJI']:
                            color = '#a2c865'
                        elif country in ['USA', 'CAN']:
                            color = '#b3b3b3'
                        elif country in ['DEU', 'AUT', 'ESP', 'FIN', 'GBR', 'ISL', 'ITA', 'LUX', 'HUN', 'SWE', 'FRA', 'NLD', 'DNK', 'IRL', 'EST', 'RUS', 'KAZ']:
                            color = '#95a3c3'
                        elif country == 'UNK':
                            color = '#000000'
                        else:
                            print('Country {} {}'.format(line, country))
                        write_file.write(line2+"\tring_color\t1\t"+color+'\n')
#                         sex = meta['gender'].values[0]
#                         age_cat = meta['age_category'].values[0]
#                         disease = meta['disease'].values[0]
                        
#                         color='#FFFFFF'
#                         if age_cat in ['newborn']:
#                             color = '#BFBFFF'
#                         elif age_cat in ['child']:
#                             color = '#979FE2'
#                         elif age_cat in['schoolage']:
#                             color = '#6E7FC6'
#                         elif age_cat in ['adult']:
#                             color = '#465FA9'
#                         elif age_cat in ['senior']:
#                             color = '#1D3F8C'
#                         else:
#                             print('Age category {} {}'.format(line, age_cat))
#                         write_file.write(line2+"\tring_color\t3\t"+color+'\n')

In [1]:
from IPython.display import Image

for tree in iglob('/shares/CIBIO-Storage/CM/scratch/users/aitor.blancomiguez/analyses/nw_metanalysis/strainphlan/output/*/RAxML_bestTree.t__SGB*.StrainPhlAn3.tre.png'):
    fig Image(filename=tree) 
    display.display(fig)
    break

NameError: name 'iglob' is not defined

In [10]:
for tree in iglob('/shares/CIBIO-Storage/CM/scratch/projects/aarre_eukaryotes/strainphlan_results/strainphlan/s__Blastocystis_sp_subtype_1/RAxML_bestTree.*.StrainPhlAn3.tre'):
    tree_n = list()
    with open(tree.replace(tree.split('/')[-1], 'graphlan_metadata.tsv'), 'w') as write_file:
        with open(tree, 'r') as read_file:
            m = read_file.readline()
            m = m.split(',')
            for n in m:
                if "'" in n:
                    tree_n.append(n.split(":")[0].split("(")[-1].split("'")[1])
                else:
                    tree_n.append(n.split(":")[0].split("(")[-1])
            for line in tree_n:
                line2 = line.replace(".", ",")
                write_file.write(line2+"\tclade_marker_edge_width\t0.2\n")

                if line in metadata['sample_id'].values.tolist():
                    meta = metadata[metadata['sample_id'] == line]
                    
                    if meta['non_westernized'].values[0] == 'NHP':
                        if line.split('__')[0] in ['LiX_2018','SrivathsanA_2015']:
                            write_file.write(line2+"\tclade_marker_color\t#FC4E07\n")
                        else:
                            write_file.write(line2+"\tclade_marker_color\t#00AFBB\n")
                            
                        write_file.write(line2+"\tclade_marker_shape\t*\n") 
                        write_file.write(line2+"\tclade_marker_size\t50\n")
            
                    
                    elif meta['non_westernized'].values[0] == 'ancient':
                        write_file.write(line2+"\tclade_marker_color\t#E7B800\n")                        
                        write_file.write(line2+"\tclade_marker_shape\t*\n") 
                        write_file.write(line2+"\tclade_marker_size\t50\n")                    
                    else:
                        write_file.write(line2+"\tclade_marker_size\t0\n")
                        if meta['non_westernized'].values[0] == 'yes':
                            write_file.write(line2+"\tring_color\t2\t#f1632a\n")
                        else:
                            write_file.write(line2+"\tring_color\t2\t#004b79\n")
                        country = meta['country'].values[0]
                        
                        if country in ['ARG', 'COL', 'PER', 'SLV']:
                            color = '#8ec06c'
                        elif country in ['CHN', 'JPN', 'IND', 'KOR', 'MNG', 'IDN', 'ISR', 'BGD']:
                            color = '#ed1b2e'
                        elif country in['ETH', 'GUI', 'MDG', 'TZA', 'GHA', 'CMR', 'LBR', 'GNB']:
                            color = '#ecb731'
                        elif country in ['FJI']:
                            color = '#537b35'
                        elif country in ['USA', 'CAN']:
                            color = '#c4dpff6'
                        elif country in ['DEU', 'AUT', 'ESP', 'FIN', 'GBR', 'ISL', 'ITA', 'LUX', 'HUN', 'SWE', 'FRA', 'NLD', 'DNK', 'IRL', 'EST', 'RUS', 'KAZ']:
                            color = '#56a0d3'
                        elif country == 'UNK':
                            color = '#000000'
                        else:
                            print('Country {} {}'.format(line, country))
                        write_file.write(line2+"\tring_color\t1\t"+color+'\n')
                        sex = meta['gender'].values[0]
                        age_cat = meta['age_category'].values[0]
                        disease = meta['disease'].values[0]
             
                        color='#FFFFFF'
                      #  if age_cat in ['newborn']:
                       #     color = '#BFBFFF'
                  #      elif age_cat in ['child']:
                   #         color = '#979FE2'
                    #    elif age_cat in['schoolage']:
                     #       color = '#6E7FC6'
                      #  elif age_cat in ['adult']:
                       #     color = '#465FA9'
                    #    elif age_cat in ['senior']:
                     #       color = '#1D3F8C'
                      #  else:
                      #      print('Age category {} {}'.format(line, age_cat))
                        color = '#ed1b2e'
                        if disease in ['healthy', 'NA']:
                            color = '#56a0d3'
                        else:
                            print('Disease {} {}'.format(line, disease))
                        write_file.write(line2+"\tring_color\t3\t"+color+'\n')
                else:
                    write_file.write(line2+"\tclade_marker_size\t0\n")
                    print('Missing {}'.format(line))

Disease CM.315_WGS STH
Disease FAT_006-22-42-0 metabolic_syndrome
Disease CCIS44757994ST-4-0 CRC
Disease YSZC12003_37102 migraine
Disease LILT_VF02_T016 adenoma;hypertension
Disease H2M513938 hypertension
Missing PNP_DietIntervention_16
Disease MH0398 T2D
Disease S462 T2D
Disease SID530743 adenoma
Disease H2M614904 hypertension
Disease M02.5-V2-stool T1D
Disease V1.FI31 IBD;perianal_fistula
Disease FAT_006-22-0-0 metabolic_syndrome
Missing SAMEA5683189
Disease YSZC12003_36675 migraine
Disease V1_UC52_1 IBD
Disease O2_UC41_2 IBD
Disease O2_UC41_0 IBD
Disease LILT_VF77_16 CRC
Disease V1_UC52_0 IBD
Disease SAMD00115028 CRC
Disease MH0355 T1D
Disease CRC_MR_SBJ12C_17 CRC
Disease FAT_023-22-84-0 metabolic_syndrome
